# GAE_HEP on Kaggle GPU

Permutation-invariant graph autoencoders for unsupervised anomaly detection on jets of the LHC Olympics 2020 R&D dataset ([github.com/raktim711/GAE_HEP](https://github.com/raktim711/GAE_HEP); results and their discussion in `RESULTS.md`).

This notebook reproduces the final models on a GPU with more seeds and adds a small scale study:

1. data: download (Zenodo record 6466204, about 3.2 GB), clustering, tables, splits, single-observable baselines;
2. **final models** with the training budget validated on CPU (5 epochs x 120k jets): the latent-4 energy-flow VAE (`configs/ef_vae.yaml`, reconstruction score) and the latent-8 energy-flow VAE (reconstruction + KL), 5 seeds each, trained with the development protocol (`gaehep dev`, validation splits only) and then evaluated on the test splits (`gaehep promote`);
3. seed ensembles, mass decorrelation, bump hunt and the weak-supervision (IAD) study on the GNN latents;
4. **scale study** (dev protocol only): wider encoders, 100 constituents per jet and longer training. On CPU, longer training *lowered* the anomaly-detection performance, so bigger is not automatically better here.

> **Before you run:** in the Kaggle session options (right-hand panel) switch **Internet on** (needed for `git clone`, `pip install` and the Zenodo download) and select a **GPU accelerator** under *Accelerator*.

In [ ]:
!nvidia-smi

import os

REPO_URL = "https://github.com/raktim711/GAE_HEP.git"
REPO_DIR = "/kaggle/working/GAE_HEP"

# clone once; skipped when the repo is already there (e.g. after a kernel restart)
if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    !git clone {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}
!pip install -q -e .

import torch

print("torch", torch.__version__, "| built for CUDA", torch.version.cuda)
print("torch.cuda.is_available():", torch.cuda.is_available())
if not torch.cuda.is_available():
    print("WARNING: no GPU visible - choose a GPU accelerator in the session options.")

In [ ]:
import os

# Kaggle: keep everything under /kaggle/working
os.environ["GAEHEP_DATA"] = "/kaggle/working/data"
os.environ["GAEHEP_RESULTS"] = "/kaggle/working/results"

# data pipeline; the shell commands inherit os.environ.
# `download` is md5-verified and resumable: re-run this cell if it gets interrupted.
!gaehep download
!gaehep preprocess --sample rnd --workers 4
!gaehep preprocess --sample qqq --workers 4
!gaehep tables --sample rnd
!gaehep tables --sample qqq
!gaehep splits
!gaehep baselines

## Final models: development protocol on GPU, then test evaluation

`gaehep dev <config> --name <name> --set key=value ...` trains with the dev budget (5 epochs x 120k jets, 20k validation jets; built into the command) and reports metrics on the **validation** splits only (`results/dev/summary.md`). `gaehep promote <dev run> <name>` then scores the same weights on the **test** splits, including the 3-prong signal that is never used for any choice.

Overrides used here: `train.device=cuda`, `train.threads=4` (data preparation stays on the CPU). Model sizes are the CPU ones, so the results are directly comparable to `RESULTS.md`; the scale study below changes them.

**YAML lists** must be passed exactly like `model_args.hidden=[128,128,128]`, without spaces, because each override is a single `--set` argument.

In [ ]:
import os
import shlex
import subprocess
import time

REPO_DIR = "/kaggle/working/GAE_HEP"

# set again so that this cell (and the next ones) also work after a kernel restart
os.environ["GAEHEP_DATA"] = "/kaggle/working/data"
os.environ["GAEHEP_RESULTS"] = "/kaggle/working/results"
os.chdir(REPO_DIR)


def gaehep(*args):
    """Run `gaehep <args>`; the log streams into the cell output, a failure stops the cell."""
    cmd = ["gaehep", *map(str, args)]
    print("$", " ".join(shlex.quote(c) for c in cmd), flush=True)
    t0 = time.time()
    subprocess.run(cmd, check=True, cwd=REPO_DIR, env={**os.environ, "PYTHONUNBUFFERED": "1"})
    print(f"  ({(time.time() - t0) / 60:.1f} min)", flush=True)


def sets(*overrides):
    out = []
    for o in overrides:
        out += ["--set", o]
    return out


GPU = ["train.device=cuda", "train.threads=4"]
VAE8 = ["model_args.variational=true", "model_args.beta=0.001", "extra_scores=[kl]"]  # on top of configs/ef_ae.yaml
SEEDS = [1, 2, 3, 4, 5]

In [ ]:
# ---- final models: dev protocol (validation) -> test evaluation; finished dev runs are reused on a re-run ----
for s in SEEDS:
    gaehep("dev", "configs/ef_vae.yaml", "--name", f"gpu_ef_vae_s{s}", *sets(*GPU, f"train.seed={s}"))
    gaehep("promote", f"gpu_ef_vae_s{s}", f"gpu_ef_vae_s{s}")
    gaehep("dev", "configs/ef_ae.yaml", "--name", f"gpu_ef_vae8_s{s}", *sets(*GPU, *VAE8, f"train.seed={s}"))
    gaehep("promote", f"gpu_ef_vae8_s{s}", f"gpu_ef_vae8_s{s}")
    gaehep("ensemble", f"gpu_ef_vae8_s{s}", f"gpu_ef_vae8_s{s}+kl", "--out", f"gpu_ef_vae8_s{s}+ens")

# ---- seed ensembles, mass decorrelation, bump hunt, weak supervision, reports ----
gaehep("ensemble", *[f"gpu_ef_vae_s{s}" for s in SEEDS], "--out", "gpu_ef_vae_seeds")
gaehep("ensemble", *[f"gpu_ef_vae8_s{s}+ens" for s in SEEDS], "--out", "gpu_ef_vae8_seeds+ens")
gaehep("qn", "gpu_ef_vae_seeds")
gaehep("bumphunt", "gpu_ef_vae_seeds", "--agg", "prod")
gaehep("weak", "gpu_ef_vae_s1")
gaehep("report")
gaehep("dev-report")

# ---- scale study (validation splits only) ----
SCALE = {
    "wide": ["model_args.hidden=[128,128,128]", "model_args.k=16"],
    "n100": ["data.n_const=100", "model_args.k=16"],
    "long": ["train.epochs=15", "train.jets_per_epoch=400000"],
}
for tag, over in SCALE.items():
    gaehep("dev", "configs/ef_vae.yaml", "--name", f"gpu_ef_vae_{tag}", *sets(*GPU, *over))
gaehep("dev-report")

In [ ]:
import os
import zipfile
from pathlib import Path

from IPython.display import FileLink, Markdown, display

RESULTS = Path(os.environ.get("GAEHEP_RESULTS", "/kaggle/working/results"))
ZIP_PATH = Path("/kaggle/working/gaehep_results.zip")

# 1) the summary table written by `gaehep report`
summary = RESULTS / "summary.md"
if summary.is_file():
    display(Markdown(summary.read_text()))
else:
    print(f"{summary} not found - run the previous cell (it ends with `gaehep report`) first.")


# 2) zip the results for download, leaving out model weights / checkpoints
def is_checkpoint(path: Path) -> bool:
    parts = [p.lower() for p in path.relative_to(RESULTS).parts]
    return path.suffix.lower() in {".pt", ".pth", ".ckpt"} or any(
        "checkpoint" in p or p.startswith("ckpt") for p in parts
    )


added = skipped = 0
with zipfile.ZipFile(ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for path in sorted(RESULTS.rglob("*")):
        if not path.is_file():
            continue
        if is_checkpoint(path):
            skipped += 1
            continue
        zf.write(path, arcname=str(Path("results") / path.relative_to(RESULTS)))
        added += 1

print(f"{ZIP_PATH}: {added} files, {ZIP_PATH.stat().st_size / 1e6:.1f} MB "
      f"({skipped} *.pt / checkpoint files left out)")

# download link (the file also shows up in the notebook's Output tab)
cwd = os.getcwd()
try:
    os.chdir(ZIP_PATH.parent)
    display(FileLink(ZIP_PATH.name))
finally:
    os.chdir(cwd)